# GLCMaeV8 Inference Pipeline

Load a trained `GLCMaeV8` checkpoint, run inference on welding robot data, and produce anomaly verdicts with phase-aware scoring.

**Data split:**
- **Train set** — used to build robot/program codebooks only
- **Val set** (normal only) — used for MAD threshold calibration
- **Test set** (50% normal / 50% anomaly, 10 types) — used for evaluation

**Pipeline:**
1. Load trained model checkpoint
2. Load train CSV → build codebooks
3. Load val CSV → preprocess into file-windows
4. Load test CSV + test_labels → preprocess into file-windows
5. Score val + test sets with `Reconf8LocalPreResult`
6. Fit threshold from val normal scores via `Reconf8LocalThresholdCreator`
7. Apply threshold to test set → evaluate per-type + overall F1 / ROC-AUC
8. Per-sensor heatmap analysis
9. Visualization
10. Summary report

In [ ]:
# %% Imports
import sys
import csv
import json
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Add anomaly-detection/src to path so import src.anomaly_detection resolves
project_root = Path.cwd().parent.parent.resolve()
src_path = project_root / "anomaly-detection" / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from src.anomaly_detection.models import (
    GLCMaeV8,
    ReconstructionModels,
)
from src.anomaly_detection.inference.recon_v8.local import (
    Reconf8LocalPreResult,
    Reconf8LocalThresholdCreator,
    Reconf8LocalApplier,
)

print("All imports OK")
print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")

All imports OK
PyTorch 2.12.0+cu130, CUDA available: True


## 1. Load Trained Model Checkpoint

Load the model state_dict + config saved by the training notebook.

In [2]:
# Path to trained checkpoint from Training-GLCReconV8 notebook
CKPT_DIR   = project_root / "artifacts" / "mae_v8_checkpoint"
MODEL_SLUG = "GLCMaeV8"
MODEL_FILE = f"model_{MODEL_SLUG}_best.pt"
META_FILE  = f"metadata_{MODEL_SLUG}_best.json"
EXTRA_FILE = CKPT_DIR / "training_extra.json"

if not (CKPT_DIR / MODEL_FILE).exists():
    print(f"Checkpoint not found at {CKPT_DIR / MODEL_FILE}")
    print("Run the Training-GLCReconV8 notebook first to train and save a model.")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = GLCMaeV8.load_from_checkpoint(
        checkpoint=str(CKPT_DIR),
        model_file=MODEL_FILE,
        metadata_file=META_FILE,
        device=str(device),
    )
    print(f"Model restored to {device}: {model.name}")
    print(f"  Params:            {sum(p.numel() for p in model.parameters()):,}")
    print(f"  num_sensors:       {model.num_sensors}")
    print(f"  seq_len:           {model.seq_len}")
    print(f"  num_robots:        {model.num_robots}")
    print(f"  num_programs:      {model.num_programs}")
    print(f"  mask_ratio:        {model.mask_ratio}")
    print(f"  use_context_emb:   {model.use_context_embeddings}")

    # Load training extras (history, codebooks, threshold)
    extras = {}
    if EXTRA_FILE.exists():
        with open(EXTRA_FILE) as f:
            extras = json.load(f)
        print(f"  Training extras loaded: history ({len(extras.get('history', {}).get('loss', []))} epochs), "
              f"robot_codebook ({len(extras.get('robot_codebook', {}))} codes), "
              f"program_codebook ({len(extras.get('program_codebook', {}))} codes)")
    robot_codebook   = extras.get("robot_codebook", {})
    program_codebook = extras.get("program_codebook", {})
    print(f"  robot_codebook:    {robot_codebook}")
    print(f"  program_codebook:  {dict(list(program_codebook.items())[:5])} ...")

Model restored to cuda: GLCMaeV8
  Params:            968,375
  num_sensors:       3
  seq_len:           256
  num_robots:        8
  num_programs:      128
  mask_ratio:        0.4
  use_context_emb:   True
  Training extras loaded: history (100 epochs), robot_codebook (4 codes), program_codebook (16 codes)
  robot_codebook:    {'0': 0, '1': 1, '2': 2, '3': 3}
  program_codebook:  {'0': 0, '1': 1, '10': 2, '11': 3, '12': 4} ...


/home/trietlm/TED-Hackathon-Agent/backend/app/anomaly_detection/layers/attention.py:97: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


## 2. Load Train + Val + Test CSV Data

Three separate CSV files (generated by Sprint 16).
Group by `cycle_id` into full-length signals (no truncation).
Build robot/program codebooks from **training extras** saved at training time.

In [ ]:
import pandas as pd

DATA_DIR        = project_root / "data" / "generated_sprint18b"
TRAIN_CSV       = DATA_DIR / "train_welding_robot.csv"
VAL_CSV         = DATA_DIR / "val_welding_robot.csv"
TEST_CSV        = DATA_DIR / "test_welding_robot.csv"
TEST_LABELS_CSV = DATA_DIR / "test_labels.csv"

SENSOR_COLS = ["wire_feed_speed_mm_min", "wire_temperature_c", "welding_current_a"]

def load_csv_to_file_records(csv_path):
    """Load CSV -> group by cycle_id -> build file_records (full signal, no truncation)."""
    print(f"Reading {csv_path}...")
    df = pd.read_csv(csv_path)
    print(f"  Rows: {len(df):,}, cycle_ids: {df['cycle_id'].nunique()}")

    records = []
    for cid, group in df.groupby("cycle_id"):
        group = group.sort_values("serial_number")
        file_arr = np.array([group[col].values for col in SENSOR_COLS], dtype=np.float32)  # [C, T] full signal
        records.append({
            "cycle_id":      cid,
            "window":        file_arr,
            "robot_code":    str(group["robot_code"].iloc[0]),
            "program_number": int(group["program_number"].iloc[0]),
            "anomaly_type":  str(group["anomaly_type"].iloc[0]),
        })
    return records

print("=" * 50)
print("LOADING DATA")
print("=" * 50)

train_records = load_csv_to_file_records(TRAIN_CSV)
val_records   = load_csv_to_file_records(VAL_CSV)
test_records  = load_csv_to_file_records(TEST_CSV)

# Use codebooks saved at training time (ensures same index mapping)
if not robot_codebook:
    print("WARNING: robot_codebook empty — rebuilding from train set")
    all_robot_codes  = sorted(set(r["robot_code"] for r in train_records))
    all_program_nums = sorted(set(str(r["program_number"]) for r in train_records))
    robot_codebook   = {code: i for i, code in enumerate(all_robot_codes)}
    program_codebook = {pn:   i for i, pn   in enumerate(all_program_nums)}
# Extend for any unseen codes in val/test
for r in val_records + test_records:
    rc, pn = str(r["robot_code"]), str(r["program_number"])
    if rc not in robot_codebook:
        robot_codebook[rc] = len(robot_codebook)
    if pn not in program_codebook:
        program_codebook[pn] = len(program_codebook)

# Load test labels
test_labels = pd.read_csv(TEST_LABELS_CSV)
label_map   = dict(zip(test_labels["cycle_id"].astype(str), test_labels["anomaly_type"]))

# Show signal length stats
train_lengths = [r["window"].shape[1] for r in train_records]
val_lengths   = [r["window"].shape[1] for r in val_records]
test_lengths  = [r["window"].shape[1] for r in test_records]
n_val_normal   = sum(1 for r in val_records)
n_test_normal  = sum(1 for r in test_records if r["anomaly_type"] == "normal")
n_test_anomaly = sum(1 for r in test_records if r["anomaly_type"] != "normal")
print(f"\nTrain: {len(train_records)} files, {len(robot_codebook)} robots, {len(program_codebook)} programs")
print(f"  Signal lengths: min={min(train_lengths)}, max={max(train_lengths)}, mean={np.mean(train_lengths):.0f}")
print(f"Val (normal):   {n_val_normal} files (min={min(val_lengths)}, max={max(val_lengths)})")
print(f"Test (normal):  {n_test_normal} files")
print(f"Test (anomaly): {n_test_anomaly} files")
print(f"Example: {test_records[0]['cycle_id']}, robot={test_records[0]['robot_code']}, "
      f"prog={test_records[0]['program_number']}, anomaly={test_records[0]['anomaly_type']}, "
      f"len={test_records[0]['window'].shape[1]}")

## 3. Build Macro-Window Records

Each file record (full-length signal) is split into overlapping windows of length `model.seq_len` with `STRIDE=64` (matching training).
Windows are accumulated with Gaussian-weight stitching in `Reconf8LocalPreResult.score_batch()`.

In [ ]:
STRIDE = 64  # match training stride

def build_pre_result_records(records_list, seq_len=None, stride=STRIDE):
    """Convert file_records -> pre_result_records for score_batch().

    Creates overlapping sub-windows of length seq_len (model's window size)
    with the given stride. Gaussian-weighted stitching is applied later in
    Reconf8LocalPreResult.score_batch().

    A tail-padding window is added when the signal doesn't evenly divide,
    matching the training notebook's behavior.
    """
    if seq_len is None:
        seq_len = model.seq_len  # model's window size (trained with 256)
    result = []
    for rec in records_list:
        window = rec["window"]   # [C, T], full signal
        C, T   = window.shape
        windows_list = []
        serials_list = []
        for start in range(0, T - seq_len + 1, stride):
            windows_list.append(window[:, start:start + seq_len])
            serials_list.append(start)
        # Tail padding (same as training)
        tail_start = ((T - seq_len) // stride) * stride + stride
        if tail_start < T:
            tail = window[:, tail_start:]
            pad = np.zeros((C, seq_len - tail.shape[1]), dtype=np.float32)
            windows_list.append(np.hstack([tail, pad]))
            serials_list.append(tail_start)
        if not windows_list:
            # Signal shorter than seq_len: pad
            pad = np.zeros((C, seq_len - T), dtype=np.float32)
            windows_list.append(np.hstack([window, pad]))
            serials_list.append(0)
        result.append({
            "robot_code":     rec["robot_code"],
            "program_number": rec["program_number"],
            "start_time":     str(rec["cycle_id"]),
            "windows":        np.stack(windows_list),
            "serials":        np.array(serials_list, dtype=np.int64),
        })
    return result

val_pre_records  = build_pre_result_records(val_records)
test_pre_records = build_pre_result_records(test_records)

print(f"Val pre-result records:  {len(val_pre_records)}")
print(f"Test pre-result records: {len(test_pre_records)}")
print(f"Val record[0]:  windows={val_pre_records[0]['windows'].shape}, "
      f"file: {len(val_pre_records[0]['windows'])} windows")
print(f"Test record[0]: windows={test_pre_records[0]['windows'].shape}")

## 4. Score Val + Test Sets

**Flow:**
1. `Reconf8LocalPreResult.score_batch()` with codebooks → per-file p95 scores (global + rampup + active)
2. `Reconf8LocalThresholdCreator` calibrated **only on val set** (100% normal)
3. Apply threshold to test set → OR-logic verdict

In [ ]:
metadata   = model.get_metadata()
state_dict = model.state_dict()
device_str = str(device)

pre_result_engine = Reconf8LocalPreResult(
    model_version=ReconstructionModels.GLC_MAE_V8,
    metadata=metadata,
    state_dict=state_dict,
    quantile=0.95,
    k_skip=0,
    device=device_str,
    robot_codebook=robot_codebook,
    program_codebook=program_codebook,
)

print("Scoring validation set (normal only)...")
val_pre_scores = pre_result_engine.score_batch(val_pre_records)

print("Scoring test set (normal + anomaly)...")
test_pre_scores = pre_result_engine.score_batch(test_pre_records)

print(f"\nVal scored:  {len(val_pre_scores)} files")
print(f"Test scored: {len(test_pre_scores)} files")
print(f"Keys in a result: {list(val_pre_scores[0].keys())[:8]}")

## 5. Val Score Distribution (Per Sensor)

Inspect per-sensor reconstruction error distributions on the **val (normal) set**. Use the percentile marks (p90/p95/p99/p99.9) to choose your manual thresholds.

In [ ]:
# ── Per-sensor val distribution + reference percentiles ─────────────────
_sensors_val = [k[len('heatmap_'):] for k in val_pre_scores[0] if k.startswith('heatmap_')]
_pq_sensor_keys = sorted(
    k for k in val_pre_scores[0]
    if k.startswith('p95_score_') and k not in {'p95_score_rampup', 'p95_score_active'}
)
_ref_qs = [50, 90, 95, 99, 99.9]

print('Reference percentiles from val (normal) set:')
header = f"{'Key':<35} " + '  '.join(f'p{q:>4.1f}' for q in _ref_qs)
print(header)
print('-' * len(header))
for sk in _pq_sensor_keys:
    arr = [r.get(sk, float('nan')) for r in val_pre_scores]
    import numpy as _np
    arr = _np.array(arr)
    arr = arr[~_np.isnan(arr)]
    pcts = [_np.percentile(arr, q) for q in _ref_qs]
    print(f'{sk:<35} ' + '  '.join(f'{p:>7.4f}' for p in pcts))
for ph in ['p95_score_rampup', 'p95_score_active']:
    if ph in val_pre_scores[0]:
        arr = _np.array([r.get(ph, float('nan')) for r in val_pre_scores])
        arr = arr[~_np.isnan(arr)]
        pcts = [_np.percentile(arr, q) for q in _ref_qs]
        print(f'{ph:<35} ' + '  '.join(f'{p:>7.4f}' for p in pcts))

# ── Plot histograms ──
import matplotlib.pyplot as _plt
n_s = len(_pq_sensor_keys)
fig, axes = _plt.subplots(1, n_s, figsize=(6 * n_s, 4), squeeze=False)
for col_i, sk in enumerate(_pq_sensor_keys):
    sensor_name = sk[len('p95_score_'):]
    arr = _np.array([r.get(sk, float('nan')) for r in val_pre_scores])
    arr = arr[~_np.isnan(arr)]
    ax = axes[0][col_i]
    ax.hist(arr, bins=30, color='steelblue', alpha=0.8, label=f'Val normal (n={len(arr)})')
    colors = ['navy', 'green', 'orange', 'crimson']
    for qi, q in enumerate([90, 95, 99, 99.9]):
        pv = _np.percentile(arr, q)
        ax.axvline(pv, color=colors[qi], ls='--', lw=1.5, label=f'p{q}={pv:.4f}')
    ax.set_title(sensor_name)
    ax.set_xlabel('p95 reconstruction error')
    ax.set_ylabel('Files')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
_plt.suptitle('Val Set — Per-Sensor Score Distribution', fontsize=13)
_plt.tight_layout()
_plt.show()
print('\nSet THRESHOLD_CONFIG in the next cell based on the above.')


## 6. Threshold Config

**Edit the cell below** — set `score_thresholds` based on the plots above.
Save and re-run the next cell to apply.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# EDIT THIS CELL — set thresholds based on the distribution plots above
# ══════════════════════════════════════════════════════════════════════

THRESHOLD_CONFIG = {
    # Set to 1e10 to disable global threshold and rely on per-sensor only
    'global_threshold': 1e10,
    'score_thresholds': {
        # Per-sensor thresholds — edit values based on the plots above
        'p95_score_wire_feed_speed':   0.20,   # ← adjust
        'p95_score_wire_temperature':  0.20,   # ← adjust
        'p95_score_welding_current':   0.40,   # ← adjust (higher baseline)
        # Phase-specific thresholds (uncomment if needed)
        # 'p95_score_rampup': 0.30,
        # 'p95_score_active': 0.30,
    },
}

# Save config for reproducibility
import json as _json
CKPT_DIR.mkdir(parents=True, exist_ok=True)
_cfg_path = CKPT_DIR / 'threshold_config.json'
with open(_cfg_path, 'w') as _f:
    _json.dump(THRESHOLD_CONFIG, _f, indent=2)
print(f'Threshold config saved to {_cfg_path}')
print('Thresholds:')
for k, v in THRESHOLD_CONFIG['score_thresholds'].items():
    print(f'  {k}: {v}')


In [ ]:
# ── Apply thresholds to test set ─────────────────────────────────────────
import warnings
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    applier = Reconf8LocalApplier(THRESHOLD_CONFIG, quantile=0.95)
    test_applied = applier.apply(test_pre_scores)
    for _w in _caught:
        print(f'\n⚠  Applier warning: {_w.message}')

print(f'Test verdict distribution:')
n_pred_anom = sum(1 for r in test_applied if r['is_file_anomaly'])
n_pred_norm = len(test_applied) - n_pred_anom
print(f'  Predicted normal:  {n_pred_norm}')
print(f'  Predicted anomaly: {n_pred_anom}')

from collections import Counter
trigger_counts = Counter()
for r in test_applied:
    for t in r.get('triggered_by', []):
        trigger_counts[t] += 1
print('Trigger distribution:')
for key, count in trigger_counts.most_common():
    print(f'  {key}: {count}')


## 7. Reference: Val Score Statistics

Statistics on the val set scores. Use alongside the distribution plots above.

In [ ]:
val_p95 = [r['p95_score'] for r in val_pre_scores]
val_p95 = [v for v in val_p95 if v == v]  # drop NaN
import numpy as _np2
arr0 = _np2.array(val_p95, dtype=float)

print('Val set p95_score statistics (global):')
for label, val in [
    ('count', int(arr0.size)),
    ('mean',  float(arr0.mean())),
    ('std',   float(arr0.std(ddof=1)) if arr0.size > 1 else 0.0),
    ('min',   float(arr0.min())),
    ('p50',   float(_np2.percentile(arr0, 50))),
    ('p95',   float(_np2.percentile(arr0, 95))),
    ('p99',   float(_np2.percentile(arr0, 99))),
    ('max',   float(arr0.max())),
]:
    print(f'{label:>8}: {val:.6f}' if label != 'count' else f'{label:>8}: {val}')

# Per-sensor
_sk_list = sorted(
    k for k in val_pre_scores[0]
    if k.startswith('p95_score_') and k not in {'p95_score_rampup', 'p95_score_active'}
)
if _sk_list:
    print('\nPer-sensor statistics:')
    print(f"{'Key':<35} {'n':>5} {'mean':>8} {'std':>8} {'p95':>8} {'p99':>8} {'max':>8}")
    print('-' * 80)
    for sk in _sk_list:
        arr = _np2.array([r.get(sk, float('nan')) for r in val_pre_scores])
        arr = arr[~_np2.isnan(arr)]
        thresh = THRESHOLD_CONFIG['score_thresholds'].get(sk, float('nan'))
        print(
            f'{sk:<35} {arr.size:>5} {arr.mean():>8.4f} '
            f'{arr.std(ddof=1):>8.4f} {_np2.percentile(arr,95):>8.4f} '
            f'{_np2.percentile(arr,99):>8.4f} {arr.max():>8.4f}'
            f'  threshold={thresh:.4f}'
        )


## 8. Evaluate: Per-Anomaly-Type + Overall Metrics

In [ ]:
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support

# ── Build ground-truth + prediction arrays ──
y_true_anomaly = np.array([
    1 if r["anomaly_type"] != "normal" else 0
    for r in test_records
])
y_pred = np.array([
    1 if r["is_file_anomaly"] else 0
    for r in test_applied
])
y_score = np.array([r["p95_score"] for r in test_applied])
anomaly_types_list = [r["anomaly_type"] for r in test_records]

# ── Per-type metrics ──
unique_types = sorted(set(anomaly_types_list))
print("=" * 70)
print("PER-ANOMALY-TYPE EVALUATION")
print("=" * 70)
print(f"{'Type':<22} {'Count':>6} {'Prec':>8} {'Recall':>8} {'F1':>8} {'ROC-AUC':>8}")
print("-" * 70)

per_type_results = {}
for at in unique_types:
    y_true_bin = np.array([1 if t == at else 0 for t in anomaly_types_list])
    if y_true_bin.sum() == 0:
        continue
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true_bin, y_pred, average="binary", zero_division=0
    )
    try:
        roc = roc_auc_score(y_true_bin, y_score)
    except ValueError:
        roc = float("nan")
    per_type_results[at] = {"precision": prec, "recall": rec, "f1": f1, "roc_auc": roc}
    print(f"{at:<22} {y_true_bin.sum():>6} {prec:>8.4f} {rec:>8.4f} {f1:>8.4f} {roc:>8.4f}")

# ── Overall (anomaly vs normal) ──
print("\n" + "=" * 70)
print("OVERALL (Anomaly vs Normal)")
print("=" * 70)
overall_prec, overall_rec, overall_f1, _ = precision_recall_fscore_support(
    y_true_anomaly, y_pred, average="binary", zero_division=0
)
try:
    overall_roc = roc_auc_score(y_true_anomaly, y_score)
except ValueError:
    overall_roc = float("nan")

tp = int(((y_true_anomaly == 1) & (y_pred == 1)).sum())
fp = int(((y_true_anomaly == 0) & (y_pred == 1)).sum())
fn = int(((y_true_anomaly == 1) & (y_pred == 0)).sum())
tn = int(((y_true_anomaly == 0) & (y_pred == 0)).sum())
accuracy = (tp + tn) / len(y_true_anomaly)

print(f"  Accuracy:  {accuracy:.4f}  ({tp + tn}/{len(y_true_anomaly)})")
print(f"  Precision: {overall_prec:.4f}")
print(f"  Recall:    {overall_rec:.4f}")
print(f"  F1 Score:  {overall_f1:.4f}")
print(f"  ROC-AUC:   {overall_roc:.4f}")
print()
print("Confusion Matrix:")
print(f"              Pred Normal   Pred Anomaly")
print(f"  True Normal   {tn:>4d}           {fp:>4d}")
print(f"  True Anomaly  {fn:>4d}           {tp:>4d}")

In [ ]:
# ── Target comparison ──
targets = {
    "spike": 0.98, "arc_blow": 0.98, "gas_blockage": 0.95,
    "thermal_drift": 0.90, "wire_stutter": 0.90, "sticky_tc": 0.85,
    "flatline": 0.85, "phase_shift": 0.80,
    "cable_wear": 0.85, "overshoot_degradation": 0.75,
}
print("=" * 70)
print("TARGET COMPARISON")
print("=" * 70)
for at, target in targets.items():
    if at in per_type_results:
        actual = per_type_results[at]["recall"]
        status = "OK" if actual >= target else "MISS"
        print(f"  {at:<22} target={target:.2f} actual={actual:.4f} {status}")

## 9. Per-Sensor Score Distribution (Test Set)

Distribution comparison: val normal vs test normal vs test anomaly, with manual threshold lines.

In [ ]:
_sensors_dist = [k[len('heatmap_'):] for k in val_pre_scores[0] if k.startswith('heatmap_')]
_sensor_score_keys_dist = sorted(
    k for k in THRESHOLD_CONFIG['score_thresholds']
    if k.startswith('p95_score_') and k not in {'p95_score_rampup', 'p95_score_active'}
)

test_is_anom = np.array([
    1 if r['anomaly_type'] != 'normal' else 0
    for r in test_records
])

if not _sensor_score_keys_dist:
    print('No per-sensor score keys found in THRESHOLD_CONFIG[score_thresholds].')
else:
    n_s = len(_sensor_score_keys_dist)
    fig, axes = plt.subplots(n_s, 3, figsize=(16, 4 * n_s), squeeze=False)

    for row_i, sk in enumerate(_sensor_score_keys_dist):
        sensor_name = sk[len('p95_score_'):]
        thresh      = THRESHOLD_CONFIG['score_thresholds'].get(sk, None)

        val_s  = np.array([r.get(sk, np.nan) for r in val_pre_scores])
        test_s = np.array([r.get(sk, np.nan) for r in test_applied])
        mask_norm = (test_is_anom == 0) & ~np.isnan(test_s)
        mask_anom = (test_is_anom == 1) & ~np.isnan(test_s)

        ax = axes[row_i][0]
        ax.hist(val_s[~np.isnan(val_s)], bins=30, alpha=0.7, color='steelblue',
                label=f'Val normal (n={(~np.isnan(val_s)).sum()})')
        ax.hist(test_s[mask_norm], bins=20, alpha=0.5, color='skyblue',
                label=f'Test normal (n={mask_norm.sum()})')
        ax.hist(test_s[mask_anom], bins=10, alpha=0.7, color='crimson',
                label=f'Test anomaly (n={mask_anom.sum()})')
        if thresh is not None:
            ax.axvline(x=thresh, color='green', ls='--', lw=2, label=f'threshold={thresh:.4f}')
        ax.set_title(f'{sensor_name} — Histogram')
        ax.set_xlabel(sk); ax.set_ylabel('Files')
        ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

        ax = axes[row_i][1]
        bp = ax.boxplot(
            [test_s[mask_norm], test_s[mask_anom]],
            labels=['Normal', 'Anomaly'], patch_artist=True,
        )
        bp['boxes'][0].set_facecolor('steelblue')
        bp['boxes'][1].set_facecolor('crimson')
        if thresh is not None:
            ax.axhline(y=thresh, color='green', ls='--', lw=2, label=f'threshold={thresh:.4f}')
            ax.legend(fontsize=7)
        ax.set_title(f'{sensor_name} — Normal vs Anomaly')
        ax.set_ylabel('Score'); ax.grid(True, alpha=0.3)

        ax = axes[row_i][2]
        sort_idx      = np.argsort(test_s)[::-1]
        sorted_scores = test_s[sort_idx]
        sorted_labels = test_is_anom[sort_idx]
        ax.plot(sorted_scores, marker='.', markersize=3, color='steelblue')
        if thresh is not None:
            ax.axhline(y=thresh, color='green', ls='--', lw=2, label=f'threshold={thresh:.4f}')
        ax.fill_between(
            range(len(sorted_scores)), sorted_scores, float(np.nanmin(sorted_scores)),
            where=(sorted_labels == 1), color='crimson', alpha=0.3, label='GT anomaly',
        )
        ax.set_title(f'{sensor_name} — Ranked')
        ax.set_xlabel('File rank (highest first)'); ax.set_ylabel('Score')
        ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    plt.suptitle('Per-Sensor Score Distribution (Test Set)', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()


## 10. Per-Sensor Heatmap Analysis

In [ ]:
# Pick one normal and one anomaly test file for heatmap comparison
normal_idx  = next(i for i, r in enumerate(test_records) if r["anomaly_type"] == "normal")
anomaly_idx = next(i for i, r in enumerate(test_records) if r["anomaly_type"] != "normal")

normal_result  = test_applied[normal_idx]
anomaly_result = test_applied[anomaly_idx]

SENSOR_NAMES = ["wire_feed_speed_mm_min", "wire_temperature_c", "welding_current_a"]
sensors_in_results = [k[len("heatmap_"):] for k in normal_result if k.startswith("heatmap_")]

fig, axes = plt.subplots(len(sensors_in_results), 2, figsize=(16, 3 * len(sensors_in_results)))
if len(sensors_in_results) == 1:
    axes = axes[None, :]

for c, s in enumerate(sensors_in_results):
    normal_heatmap  = np.array(normal_result.get(f"heatmap_{s}", []))
    anomaly_heatmap = np.array(anomaly_result.get(f"heatmap_{s}", []))

    t_normal  = np.arange(len(normal_heatmap))
    t_anomaly = np.arange(len(anomaly_heatmap))

    axes[c, 0].plot(t_normal,  normal_heatmap,  color="steelblue", lw=1.5)
    axes[c, 0].set_title(f"{s} — Normal file")
    axes[c, 0].set_ylabel("Recon error")
    axes[c, 0].grid(True, alpha=0.3)

    axes[c, 1].plot(t_anomaly, anomaly_heatmap, color="crimson", lw=1.5)
    axes[c, 1].set_title(f"{s} — Anomaly file ({anomaly_result['anomaly_type']})"
                          if hasattr(anomaly_result, 'get') and 'anomaly_type' in anomaly_result
                          else f"{s} — Anomaly file")
    axes[c, 1].set_ylabel("Recon error")
    axes[c, 1].grid(True, alpha=0.3)

plt.suptitle(
    f"Normal ({normal_result['robot_code']}/{normal_result['program_number']}) vs "
    f"Anomaly ({anomaly_result['robot_code']}/{anomaly_result['program_number']})",
    fontsize=12,
)
plt.tight_layout()
plt.show()

print(f"Normal  file: p95={normal_result['p95_score']:.4f}  is_anomaly={normal_result['is_file_anomaly']}")
anomaly_score_key = "p95_score"
print(f"Anomaly file: p95={anomaly_result[anomaly_score_key]:.4f}  is_anomaly={anomaly_result['is_file_anomaly']}")
print(f"  triggered_by: {anomaly_result.get('triggered_by', [])}")

## 11. Phase-Score Distribution

In [ ]:
if "p95_score_rampup" in test_applied[0]:
    rampup_scores = np.array([r.get("p95_score_rampup", 0.0) for r in test_applied])
    active_scores = np.array([r.get("p95_score_active", 0.0) for r in test_applied])

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    for ax, scores, title in [
        (axes[0], rampup_scores, "Rampup Phase Score"),
        (axes[1], active_scores, "Active Phase Score"),
    ]:
        ax.hist(scores[test_is_anom == 0], bins=20, alpha=0.7,
                color="steelblue", label="Normal")
        ax.hist(scores[test_is_anom == 1], bins=10, alpha=0.7,
                color="crimson", label="Anomaly")
        key = "p95_score_rampup" if "Rampup" in title else "p95_score_active"
        if key in tc.thresholds:
            ax.axvline(x=tc.thresholds[key], color="green", ls="--", lw=2,
                       label=f"threshold={tc.thresholds[key]:.4f}")
        ax.set_xlabel("Score")
        ax.set_ylabel("Files")
        ax.set_title(title)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("Phase scores not available in pre-results (check score_batch output keys)")

## 12. Summary Report

In [ ]:
print("=" * 70)
print("GLCMaeV8 INFERENCE SUMMARY")
print("=" * 70)
print(f"  Model:              {model.name}")
print(f"  Checkpoint:         {CKPT_DIR / MODEL_FILE}")
print(f"  Device:             {device}")
print()
print("--- DATA SPLIT ---")
print(f"  Train files:        {len(train_records)}")
print(f"  Val files (normal): {len(val_records)}")
print(f"  Test files:         {len(test_records)}")
print(f"    Normal:  {n_test_normal}")
print(f"    Anomaly: {n_test_anomaly} ({len(unique_types) - 1} types)")
print()
print("--- THRESHOLDS (from val set) ---")
for key, thresh in tc.thresholds.items():
    med   = tc._medians.get(key, float('nan'))
    mad_v = tc._mads.get(key, float('nan'))
    print(f"  {key}: {thresh:.6f}  (median={med:.4f}, MAD={mad_v:.4f})")
print()
print("--- OVERALL METRICS (test set) ---")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {overall_prec:.4f}")
print(f"  Recall:    {overall_rec:.4f}")
print(f"  F1 Score:  {overall_f1:.4f}")
print(f"  ROC-AUC:   {overall_roc:.4f}")
print()
print("--- PER-TYPE RECALL ---")
for at in sorted(per_type_results.keys()):
    r = per_type_results[at]
    target = targets.get(at, None)
    status = "OK" if target and r["recall"] >= target else "MISS" if target else ""
    target_str = f" (target {target:.0%})" if target else ""
    print(f"  {at:<22} recall={r['recall']:.2%}  f1={r['f1']:.2%}  roc={r['roc_auc']:.2%}{target_str} {status}")
print()
print("--- CONFUSION MATRIX ---")
print(f"              Pred Normal   Pred Anomaly")
print(f"  True Normal   {tn:>4d}           {fp:>4d}")
print(f"  True Anomaly  {fn:>4d}           {tp:>4d}")
print("=" * 70)